# Notebook 1 — subprocess básico
**Taller: Python + argparse + subprocess → preparación para OpenTelemetry**

En este notebook aprendemos a lanzar procesos del sistema operativo desde Python usando `subprocess`. Al final sabrás:
- Qué es `subprocess` y cuándo usarlo
- Ejecutar comandos simples y capturar su salida
- Distinguir `stdout` de `stderr`
- Manejar códigos de retorno
- Ejecutar scripts `.sh` con argumentos

---
## 1.1  ¿Qué es `subprocess`?

`subprocess` es el módulo estándar de Python para **lanzar y controlar procesos externos**.
Piénsalo como "abrir una terminal desde Python" sin salir del intérprete.

```
Python  ──subprocess.run()──►  proceso del SO
           ◄── stdout / stderr / returncode ──
```

Casos de uso típicos:
- Ejecutar scripts bash de administración
- Llamar a herramientas CLI (ffmpeg, git, curl…)
- Orquestar pipelines de datos
- Recolectar métricas del sistema (base de OpenTelemetry)

In [ ]:
# Importa subprocess y sys, luego imprime la versión de Python


---
## 1.2  `subprocess.run()` — la función principal

Firma simplificada:
```python
resultado = subprocess.run(
    args,           # lista de strings o string
    capture_output, # True para capturar stdout/stderr
    text,           # True para obtener strings, no bytes
    check,          # True para lanzar excepción si falla
    cwd,            # directorio de trabajo del proceso
    env,            # variables de entorno
    timeout,        # segundos máximos
)
```
Retorna un objeto `CompletedProcess` con `.stdout`, `.stderr`, `.returncode`.

### Ejercicio 1 — Primer comando

Usa `subprocess.run()` para ejecutar `echo "Hola desde subprocess"`.  
Captura la salida e imprime `.stdout`, `.stderr` y `.returncode`.

In [2]:
import subprocess

# Ejercicio 1# Ejecuta el comando echo y captura su salida
resultado = subprocess.run(["echo", "Hola desde subprocess"], capture_output=True, text=True)

# Imprime cada uno de los atributos del objeto CompletedProcess
print("stdout:", resultado.stdout)
print("stderr:", resultado.stderr)
print("returncode:", resultado.returncode)

stdout: Hola desde subprocess

stderr: 
returncode: 0


### Ejercicio 2 — Bytes vs texto

Ejecuta el comando `date` dos veces:  
- Una vez **sin** `text=True` y muestra el tipo y valor de `.stdout`  
- Una vez **con** `text=True` y muestra el tipo y valor de `.stdout`  

¿Qué diferencia observas?

In [4]:
# Ejercicio # Primera vez: sin text=True (Muestra bytes)
res_bytes = subprocess.run(["date"], capture_output=True)
print("Sin text=True -> Tipo:", type(res_bytes.stdout), "| Valor:", res_bytes.stdout)

# Segunda vez: con text=True (Muestra un string estructurado)
res_texto = subprocess.run(["date"], capture_output=True, text=True)
print("Con text=True -> Tipo:", type(res_texto.stdout), "| Valor:", res_texto.stdout)

# Respuesta analítica solicitada:
# Sin 'text=True', la salida se recibe como un objeto de tipo bytes (prefijo b'').
# Con 'text=True', Python decodifica automáticamente esos bytes a una cadena de texto (string) legible.
2


Sin text=True -> Tipo: <class 'bytes'> | Valor: b'Fri Aug 14 01:40:56 AM UTC 2026\n'
Con text=True -> Tipo: <class 'str'> | Valor: Fri Aug 14 01:40:56 AM UTC 2026



2

---
## 1.3  stdout vs stderr y returncode

| Canal        | Para qué sirve                          |
|--------------|-----------------------------------------|
| `stdout`     | Salida normal del programa              |
| `stderr`     | Mensajes de error / diagnóstico         |
| `returncode` | 0 = éxito, cualquier otro = fallo       |

### Ejercicio 3 — Capturar ambos canales

Usa `bash -c` para ejecutar un script inline que:  
- Escriba `"esto va a stdout"` en stdout  
- Escriba `"esto va a stderr"` en stderr (pista: `echo "msg" >&2`)  
- Escriba `"ultima linea stdout"` en stdout  

Captura y muestra cada canal por separado junto con el `returncode`.

In [5]:
# Ejercicio 3# Comando multilínea usando el shell bash para direccionar flujos
script_inline = 'echo "esto va a stdout"\necho "esto va a stderr" >&2\necho "ultima linea stdout"'
resultado = subprocess.run(["bash", "-c", script_inline], capture_output=True, text=True)

# Muestra canales separados y estado de finalización
print("Canal STDOUT:\n", resultado.stdout)
print("Canal STDERR:\n", resultado.stderr)
print("Código de retorno:", resultado.returncode)



Canal STDOUT:
 esto va a stdout
ultima linea stdout

Canal STDERR:
 esto va a stderr

Código de retorno: 0


### Ejercicio 4 — Función `ejecutar()` con resumen

Escribe una función `ejecutar(comando: list[str]) -> None` que:  
1. Ejecute el comando  
2. Imprima `[OK]` o `[FALLO (código N)]` según el `returncode`  
3. Imprima el `stdout` y `stderr` si no están vacíos  

Pruébala con `ls /tmp` (debería funcionar) y `ls /ruta/que/no/existe` (debería fallar).

In [8]:
# Ejercicio 4
def ejecutar(comando: list[str]) -> None:
    # Ejecuta el comando guardando salidas
    resultado = subprocess.run(comando, capture_output=True, text=True)

    # Evalúa el código de retorno
    if resultado.returncode == 0:
        print("[OK]")
    else:
        print(f"[FALLO (código {resultado.returncode})]")

    # Muestra los flujos si contienen información válida
    if resultado.stdout.strip():
        print("Salida (stdout):\n", resultado.stdout)
    if resultado.stderr.strip():
        print("Error (stderr):\n", resultado.stderr)

# Pruebas de verificación solicitadas
print("--- Probando comando exitoso ---")
ejecutar(["ls", "/tmp"])

print("\n--- Probando comando erróneo ---")
ejecutar(["ls", "/ruta/inexistente"])

--- Probando comando exitoso ---
[OK]
Salida (stdout):
 initgoogle_syslog_dir.0
language_service.3d749f740088.root.log.INFO.20260814-013842.1375
language_service.INFO
pyright-1381-3dA88RDdErgA
python-languageserver-cancellation


--- Probando comando erróneo ---
[FALLO (código 2)]
Error (stderr):
 ls: cannot access '/ruta/inexistente': No such file or directory



### Ejercicio 5 — `check=True` y excepciones

Ejecuta `ls /ruta/inexistente` con `check=True` dentro de un bloque `try/except`.  
Captura `subprocess.CalledProcessError` e imprime el código de retorno y el mensaje de error.

In [9]:
# Ejercicio 5
try:
    # Forzar la excepción usando el parámetro check=True
    subprocess.run(["ls", "/ruta/inexistente"], check=True, capture_output=True, text=True)
except subprocess.CalledProcessError as e:
    # Captura y aislamiento de los datos del error del sistema
    print("Código de retorno capturado (e.returncode):", e.returncode)
    print("Mensaje de error capturado (e.stderr):", e.stderr)


Código de retorno capturado (e.returncode): 2
Mensaje de error capturado (e.stderr): ls: cannot access '/ruta/inexistente': No such file or directory



---
## 1.4  Ejecutar scripts `.sh`

Los scripts deben tener permisos de ejecución (`chmod +x`).  
Desde Python podemos otorgárselos programáticamente con `pathlib` y `stat`.

### Ejercicio 6 — Dar permisos de ejecución

Usa `Path` y `stat` para dar permisos de ejecución (`+x`) a todos los archivos `.sh` de la carpeta `scripts/`.  
Imprime el nombre de cada archivo que procesas.

In [29]:
# Crea el directorio 'scripts' si no existe
!mkdir -p scripts

In [30]:
# Crea el script 'saludo.sh'
%%writefile scripts/saludo.sh
#!/bin/bash
NOMBRE=$1
IDIOMA=$2

if [ -z "$IDIOMA" ]; then
    IDIOMA="es"
fi

case "$IDIOMA" in
    es)
        echo "¡Hola, $NOMBRE!"
        ;;
    en)
        echo "Hello, $NOMBRE!"
        ;;
    fr)
        echo "Bonjour, $NOMBRE!"
        ;;
    *)
        echo "Idioma '$IDIOMA' no soportado para $NOMBRE."
        exit 1 # Indicar un error para idioma no soportado
        ;;
esac

Overwriting scripts/saludo.sh


In [28]:
import stat
from pathlib import Path

SCRIPTS = Path("scripts")

# Ejercicio 6
import stat
from pathlib import Path

SCRIPTS = Path("scripts")

# Bucle interactivo sobre la colección de archivos encontrados
for archivo_sh in SCRIPTS.glob("*.sh"):
    # Suma binaria de permisos de ejecución para Dueño, Grupo y Otros
    permisos_actuales = archivo_sh.stat().st_mode
    archivo_sh.chmod(permisos_actuales | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    print(f"Permisos aplicados correctamente a: {archivo_sh}")



Permisos aplicados correctamente a: scripts/saludo.sh


### Ejercicio 7 — Ejecutar `saludo.sh`

Escribe una función `saludar(nombre: str, idioma: str = "es") -> str` que:  
1. Ejecute `scripts/saludo.sh` pasando `nombre` e `idioma` como argumentos  
2. Retorne la salida limpia (sin `\n` al final)  

Pruébala con al menos 3 combinaciones distintas (idiomas: `es`, `en`, `fr` y uno no soportado).

In [31]:
import subprocess

def saludar(nombre: str, idioma: str = "es") -> str:
    # Construcción de la llamada pasándole los parámetros dinámicos
    resultado = subprocess.run(["scripts/saludo.sh", nombre, idioma], capture_output=True, text=True)
    return resultado.stdout.strip()

# Set de pruebas requerido por la guía
print(saludar("Ana"))
print(saludar("John", "en"))
print(saludar("Marie", "fr"))

# Prueba con idioma no soportado
print("\n--- Prueba de idioma no válido ---")
print(saludar("Carlos", "it"))

¡Hola, Ana!
Hello, John!
Bonjour, Marie!

--- Prueba de idioma no válido ---
Idioma 'it' no soportado para Carlos.


### Ejercicio 8 — Ejecutar `info_sistema.sh` y parsear la salida

El script produce líneas con formato `Clave  : Valor`.  
1. Ejecuta `info_sistema.sh`  
2. Recorre las líneas de la salida  
3. Para cada línea que contenga `:` y no empiece con `===`, extrae la clave y el valor  
4. Guárdalos en un diccionario e imprímelo

In [41]:
# Ejercicio 8
import subprocess

# Antes de ejecutar este ejercicio, asegúrate de haber creado el script 'scripts/info_sistema.sh'.
# Si aún no lo has hecho, puedes ejecutar la siguiente celda sugerida por el agente para crearlo.

# Ejecución limpia del script recolector
resultado = subprocess.run(["scripts/info_sistema.sh"], capture_output=True, text=True)
diccionario_sistema = {}

# Procesamiento por bloques de texto planos línea por línea
for linea in resultado.stdout.splitlines():
    # Validaciones lógicas: debe tener divisor ':' y saltarse cabeceras de diseño '==='
    if ":" in linea and not linea.startswith("==="):
        # Partición limpia utilizando la primera coincidencia del separador
        clave, _, valor = linea.partition(":")
        diccionario_sistema[clave.strip()] = valor.strip()

print("Diccionario procesado:", diccionario_sistema)

Diccionario procesado: {'Hostname': '3d749f740088', 'Kernel': 'Linux', 'Versión Kernel': '6.6.122+', 'Arquitectura': 'x86_64', 'Usuario': 'root', 'Uptime': 'up 23 minutes', 'Carga media': '0.30', 'Memoria total': '12Gi', 'Memoria libre': '8.8Gi', 'Discos': '/dev/root: 2.0G, usado: 65%', '/dev/sda1': '114G, usado: 19%', 'IP pública': '34.86.117.225'}


### Crea el script `info_sistema.sh`

In [38]:
%%writefile scripts/info_sistema.sh
#!/bin/bash

# Simula la recolección de información del sistema

echo "=== Información del Sistema ==="
echo "Hostname: $(hostname)"
echo "Kernel: $(uname -s)"
echo "Versión Kernel: $(uname -r)"
echo "Arquitectura: $(uname -m)"
echo "Usuario: $(whoami)"
echo "Uptime: $(uptime -p)"
echo "Carga media: $(uptime | awk -F'load average:' '{print $2}' | cut -d',' -f1)"
echo "Memoria total: $(free -h | awk '/Mem:/ {print $2}')"
echo "Memoria libre: $(free -h | awk '/Mem:/ {print $4}')"
echo "Discos: $(df -h | grep '^/dev/' | awk '{print $1 ": " $2 ", usado: " $5}')"
echo "IP pública: $(curl -s ifconfig.me)"
echo "=== Fin Información ==="

Overwriting scripts/info_sistema.sh


In [40]:
# Otorgar permisos de ejecución al script info_sistema.sh
import subprocess

subprocess.run(['chmod', '+x', 'scripts/info_sistema.sh'], check=True)
print('Permisos de ejecución otorgados a scripts/info_sistema.sh')

Permisos de ejecución otorgados a scripts/info_sistema.sh


---
## 1.5  `timeout` — protege tu programa

Sin timeout, un comando colgado bloquea Python indefinidamente.  
Usa el parámetro `timeout=N` (segundos) para evitarlo.

### Ejercicio 9 — Timeout

Ejecuta `sleep 10` con un `timeout` de 2 segundos.  
Captura `subprocess.TimeoutExpired` e imprime un mensaje indicando cuántos segundos se esperaron.

In [43]:
# Ejercicio 9
import subprocess

try:
    # Límite estricto de tiempo configurado en 2 segundos
    subprocess.run(["sleep", "10"], timeout=2, capture_output=True, text=True)
except subprocess.TimeoutExpired as e:
    # Extracción de la métrica de tiempo configurada en el objeto contenedor del fallo
    print(f"El proceso fue cancelado. Se configuró un límite de (e.timeout): {e.timeout} segundos.")

El proceso fue cancelado. Se configuró un límite de (e.timeout): 2 segundos.


---
## Reto del notebook 1

Escribe una función `info_dict() -> dict` que:  
1. Ejecute `info_sistema.sh` con un timeout de 5 segundos  
2. Retorne un `dict` con las claves en minúsculas y sin espacios (p. ej. `"hostname"`, `"usuario"`)  
3. Agregue la clave `"timestamp"` con `datetime.now().isoformat()`  
4. Si el script falla por cualquier razón, retorne `{"error": mensaje_de_error}`  

Verifica que el resultado contenga la clave `"timestamp"`.

In [44]:
from datetime import datetime

def info_dict() -> dict:
    ...

# resultado = info_dict()
# assert "timestamp" in resultado
# print(resultado)
from datetime import datetime
import subprocess

def info_dict() -> dict:
    try:
        # Ejecuta el script de manera segura con el límite solicitado
        resultado = subprocess.run(
            ["scripts/info_sistema.sh"],
            timeout=5,
            capture_output=True,
            text=True,
            check=True
        )

        datos = {}
        # Procesar la salida línea a línea para limpiar y transformar
        for linea in resultado.stdout.splitlines():
            if ":" in linea and not linea.startswith("==="):
                clave, _, valor = linea.partition(":")
                # Claves normalizadas: a minúsculas y eliminando espacios internos/externos
                clave_limpia = clave.strip().lower().replace(" ", "")
                datos[clave_limpia] = valor.strip()

        # Inserción de la métrica temporal ISO 8601 exigida
        datos["timestamp"] = datetime.now().isoformat()
        return datos

    except Exception as e:
        # Captura genérica de cualquier fallo en la ejecución o procesamiento
        # Si es un error de subprocess, extrae stderr; si no, el mensaje de excepción clásico
        mensaje_error = getattr(e, 'stderr', str(e)) or str(e)
        return {"error": mensaje_error.strip()}

# Celda de validación (Assert) para confirmar el éxito del reto
resultado_reto = info_dict()
print("Resultado final obtenido:", resultado_reto)

# Verifica que la firma del timestamp exista y valide el estado
assert "timestamp" in resultado_reto, "El reto falló: Falta la clave 'timestamp' en el diccionario"
print("\n¡Reto completado con éxito! El assert pasó la validación sin errores.")


Resultado final obtenido: {'hostname': '3d749f740088', 'kernel': 'Linux', 'versiónkernel': '6.6.122+', 'arquitectura': 'x86_64', 'usuario': 'root', 'uptime': 'up 25 minutes', 'cargamedia': '0.21', 'memoriatotal': '12Gi', 'memorialibre': '8.8Gi', 'discos': '/dev/root: 2.0G, usado: 65%', '/dev/sda1': '114G, usado: 19%', 'ippública': '34.86.117.225', 'timestamp': '2026-08-14T01:58:57.891583'}

¡Reto completado con éxito! El assert pasó la validación sin errores.
